# SE4050 Deep Learning Assignment
## Notebook 3: Custom CNN — Baseline Model
### Brain Tumor MRI Classification — Model 1 of 4

**Architecture**: Custom convolutional neural network trained from scratch  
**Role**: Baseline model; establishes the performance floor against which transfer learning models are compared  
**Justification**: Unlike transfer learning architectures, this model has no prior knowledge of ImageNet features.
Training from random initialisation on the MRI dataset alone reveals the data requirements
for learning domain-specific visual patterns. The architecture follows established design
principles: progressively deeper filters (32 -> 64 -> 128 -> 256), batch normalisation
for training stability, and global average pooling to reduce parameter count and overfitting.

**Prerequisite**: Execute `02_Preprocessing.ipynb` to generate the `preprocessed_data/` directory.

## Section 0: Environment Setup

In [ ]:
# ---------------------------------------------------------------------------
# PyTorch Keras backend must be set before any Keras import.
# This enables GPU acceleration via CUDA 12.8 on Windows,
# circumventing the TensorFlow GPU limitation on Windows >= 2.11.
# ---------------------------------------------------------------------------
import os
os.environ['KERAS_BACKEND'] = 'torch'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # Suppress TensorFlow C++ logging

# Standard library
import json, time, random, warnings
from pathlib import Path

# Numerical computing
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch — deep learning framework
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Evaluation metrics from scikit-learn
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    accuracy_score, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Reproducibility: fix all random seeds to ensure consistent training runs
# ---------------------------------------------------------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# ---------------------------------------------------------------------------
# Device selection: prefer CUDA GPU for accelerated training
# ---------------------------------------------------------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU model      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM available : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ---------------------------------------------------------------------------
# Directory and hyperparameter configuration
# ---------------------------------------------------------------------------
DATA_DIR  = Path('preprocessed_data')
MODEL_DIR = Path('saved_models') / 'CNN'
RES_DIR   = Path('results')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(exist_ok=True)

# Training hyperparameters
BATCH_SIZE  = 32     # Mini-batch size; larger batches use more VRAM
MAX_EPOCHS  = 60     # Upper bound on epochs; early stopping will typically terminate earlier
INIT_LR     = 1e-4   # Initial learning rate for Adam
MODEL_NAME  = 'CNN'

print(f'\nHyperparameters:')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Max epochs  : {MAX_EPOCHS}')
print(f'  Initial LR  : {INIT_LR}')

## Section 1: Load Preprocessed Data

In [ ]:
# ---------------------------------------------------------------------------
# Load the NumPy arrays produced by 02_Preprocessing.ipynb.
# These arrays are normalised to [0, 1] and contain the 70/15/15 split.
# ---------------------------------------------------------------------------
X_train = np.load(DATA_DIR / 'X_train.npy')  # shape: (N_train, 224, 224, 3), float32
y_train = np.load(DATA_DIR / 'y_train.npy')  # shape: (N_train,), int64
X_val   = np.load(DATA_DIR / 'X_val.npy')
y_val   = np.load(DATA_DIR / 'y_val.npy')
X_test  = np.load(DATA_DIR / 'X_test.npy')
y_test  = np.load(DATA_DIR / 'y_test.npy')

# Class metadata
class_weights_arr = np.load(DATA_DIR / 'class_weights.npy')  # float32 per-class weights
CLASS_NAMES       = np.load(DATA_DIR / 'class_names.npy', allow_pickle=True).tolist()
NUM_CLASSES       = len(CLASS_NAMES)

# Move class weights to the training device (required by PyTorch loss function)
class_weights_tensor = torch.FloatTensor(class_weights_arr).to(DEVICE)

# Consistent class colour palette for all visualisations
CLASS_COLORS = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']

print(f'Arrays loaded:')
print(f'  X_train : {X_train.shape}  y_train : {y_train.shape}')
print(f'  X_val   : {X_val.shape}  y_val   : {y_val.shape}')
print(f'  X_test  : {X_test.shape}  y_test  : {y_test.shape}')
print(f'  Classes : {CLASS_NAMES}')
print(f'  Weights : {class_weights_arr.round(4)}')

## Section 2: Dataset and DataLoader

A custom PyTorch Dataset class wraps the NumPy arrays and applies data augmentation
to training images at load time. Augmentation is applied stochastically, so each epoch
sees slightly different versions of the training images, which acts as a regulariser.
Validation and test images receive only conversion transforms — no augmentation.

In [ ]:
class MRIDataset(Dataset):
    """
    PyTorch Dataset wrapping pre-loaded NumPy arrays for brain MRI images.

    Training images receive stochastic augmentation to increase effective
    dataset diversity and reduce overfitting. Validation and test images
    are converted to tensors without any augmentation.

    Parameters
    ----------
    X       : float32 array of shape (N, H, W, 3) with values in [0, 1]
    y       : int64  label array of shape (N,)
    augment : if True, apply random training-time transforms
    """

    # Augmentation pipeline applied only during training.
    # Parameters are chosen to reflect realistic MRI acquisition variability:
    #   - Rotation: patients may not be perfectly aligned in the scanner
    #   - Horizontal/vertical flips: anatomical symmetry allows both orientations
    #   - Affine shear and translate: simulate slight patient movement
    #   - Brightness jitter: models variation in MRI contrast settings
    TRAIN_TRANSFORMS = T.Compose([
        T.ToPILImage(),
        T.RandomRotation(degrees=40),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.3),
        T.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=20, scale=(0.8, 1.2)),
        T.ColorJitter(brightness=0.2, contrast=0.15),
        T.ToTensor(),   # Converts PIL (H, W, C) [0, 255] -> Tensor (C, H, W) [0, 1]
    ])

    # Validation / test: only tensor conversion; no random transforms
    EVAL_TRANSFORMS = T.Compose([
        T.ToPILImage(),
        T.ToTensor(),
    ])

    def __init__(self, X: np.ndarray, y: np.ndarray, augment: bool = False):
        # Scale back to [0, 255] uint8 for PIL-compatible transforms,
        # since T.ToPILImage() expects uint8 input.
        self.X       = (X * 255).astype(np.uint8)
        self.y       = y.astype(np.int64)
        self.transform = self.TRAIN_TRANSFORMS if augment else self.EVAL_TRANSFORMS

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        # Apply the configured transform and return (image_tensor, label)
        return self.transform(self.X[idx]), self.y[idx]


# Create a seeded Generator for the training DataLoader shuffle to maintain reproducibility
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    MRIDataset(X_train, y_train, augment=True),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, generator=g, pin_memory=True
)
val_loader = DataLoader(
    MRIDataset(X_val, y_val, augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    MRIDataset(X_test, y_test, augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)

print(f'DataLoaders created:')
print(f'  Training   : {len(train_loader)} batches  (x{BATCH_SIZE} images each)')
print(f'  Validation : {len(val_loader)} batches')
print(f'  Test       : {len(test_loader)} batches')

## Section 3: Custom CNN Architecture

### Design Rationale

The architecture comprises four convolutional blocks with progressively increasing filter
depth (32, 64, 128, 256). Each block applies:
- Two 3x3 convolutions to extract spatial features at the current resolution
- Batch normalisation to stabilise gradient flow and reduce internal covariate shift
- ReLU activation for non-linearity
- 2x2 max-pooling to halve spatial dimensions and introduce limited translation invariance

Global Average Pooling (GAP) replaces a flattened fully connected layer,
reducing the parameter count significantly while providing better regularisation.
GAP enforces that each feature map channel corresponds to one spatial concept,
which improves interpretability and reduces overfitting on small medical datasets.

In [ ]:
class ConvBlock(nn.Module):
    """
    A single convolutional block consisting of:
        Conv2D(in_ch -> out_ch, 3x3) -> BN -> ReLU
        Conv2D(out_ch -> out_ch, 3x3) -> BN -> ReLU
        MaxPool2D(2x2)
        Dropout2D for spatial regularisation
    """
    def __init__(self, in_channels: int, out_channels: int, dropout_p: float = 0.25):
        super().__init__()
        self.block = nn.Sequential(
            # First convolution: extract initial features at this depth
            nn.Conv2d(in_channels,  out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            # Second convolution: refine features without changing spatial dimensions
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            # Halve spatial dimensions; reduce computation for subsequent layers
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Spatial dropout drops entire feature maps randomly to regularise
            nn.Dropout2d(p=dropout_p),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class BrainTumorCNN(nn.Module):
    """
    Custom CNN for brain tumor MRI classification (4 classes).

    Architecture summary:
        Input:  (B, 3, 224, 224)
        Block1: Conv(3->32)   -> BN -> ReLU x2 -> MaxPool -> Drop2D  => (B, 32,  112, 112)
        Block2: Conv(32->64)  -> BN -> ReLU x2 -> MaxPool -> Drop2D  => (B, 64,   56,  56)
        Block3: Conv(64->128) -> BN -> ReLU x2 -> MaxPool -> Drop2D  => (B, 128,  28,  28)
        Block4: Conv(128->256)-> BN -> ReLU x2 -> MaxPool -> Drop2D  => (B, 256,  14,  14)
        GAP:    AdaptiveAvgPool(1x1)                                  => (B, 256,   1,   1)
        Head:   Flatten -> FC(256->256) -> BN -> ReLU -> Drop(0.5)
                        -> FC(256->128) -> ReLU -> Drop(0.4)
                        -> FC(128->4)
        Output: (B, 4)  raw logits (softmax applied by CrossEntropyLoss)
    """
    def __init__(self, num_classes: int = 4):
        super().__init__()

        # Four convolutional blocks with increasing filter depth
        self.conv_blocks = nn.Sequential(
            ConvBlock(3,   32,  dropout_p=0.25),
            ConvBlock(32,  64,  dropout_p=0.25),
            ConvBlock(64,  128, dropout_p=0.25),
            ConvBlock(128, 256, dropout_p=0.25),
        )

        # Global average pooling collapses spatial dimensions to 1x1
        # This makes the architecture input-size agnostic and reduces overfitting
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Fully connected classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.50),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(128, num_classes),  # Raw logits; no softmax (handled by loss)
        )

        # Initialise weights using He (Kaiming) initialisation,
        # which is appropriate for ReLU activation functions
        self._init_weights()

    def _init_weights(self):
        """Apply Kaiming (He) normal initialisation to Conv2d and Linear layers."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias,   0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv_blocks(x)
        x = self.gap(x)
        return self.classifier(x)


# Instantiate the model and transfer to the selected compute device
model = BrainTumorCNN(num_classes=NUM_CLASSES).to(DEVICE)

# Report parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model architecture: BrainTumorCNN')
print(f'Total parameters   : {total_params:,}')
print(f'Trainable params   : {trainable_params:,}')

# Forward-pass sanity check with a random dummy batch
dummy_input = torch.randn(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    dummy_output = model(dummy_input)
print(f'Forward pass check : {tuple(dummy_input.shape)} -> {tuple(dummy_output.shape)}')
print('Architecture verified successfully.')

## Section 4: Training Infrastructure

Three components are defined here to support robust training:
1. `train_epoch` — performs one full pass over the training set and returns loss and accuracy
2. `evaluate` — evaluates the model on a loader without gradient computation
3. `EarlyStopping` — monitors validation loss and halts training when no improvement
   is observed for `patience` consecutive epochs, saving the best checkpoint

In [ ]:
def train_epoch(model: nn.Module, loader: DataLoader,
                criterion: nn.Module, optimizer: torch.optim.Optimizer,
                device: torch.device) -> tuple:
    """
    Execute one forward and backward pass over the entire training DataLoader.

    Returns
    -------
    avg_loss : float  — mean cross-entropy loss per sample
    accuracy : float  — fraction of correctly classified samples
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()

        # Gradient clipping prevents exploding gradients on difficult batches
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(dim=1) == y_batch).sum().item()
        total      += len(y_batch)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader,
             criterion: nn.Module, device: torch.device) -> tuple:
    """
    Evaluate the model on a given DataLoader without updating weights.
    The @torch.no_grad() decorator disables gradient tracking for efficiency.

    Returns
    -------
    avg_loss : float
    accuracy : float
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)
        logits  = model(X_batch)
        loss    = criterion(logits, y_batch)
        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(dim=1) == y_batch).sum().item()
        total      += len(y_batch)

    return total_loss / total, correct / total


class EarlyStopping:
    """
    Monitor validation loss and signal when training should stop.

    Training is halted if the validation loss does not improve by more
    than `min_delta` over `patience` consecutive epochs. The best model
    weights are saved at `model_path` whenever an improvement is recorded.

    Parameters
    ----------
    patience  : int   — number of epochs without improvement before stopping
    min_delta : float — minimum absolute improvement to qualify as an improvement
    """
    def __init__(self, patience: int = 12, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float('inf')
        self.should_stop = False

    def __call__(self, val_loss: float) -> bool:
        """
        Check the current validation loss against the best recorded.
        Returns True if a new best was achieved (checkpoint should be saved).
        """
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
            return True    # Improved: save checkpoint
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False   # No improvement


print('Training utilities (train_epoch, evaluate, EarlyStopping) defined successfully.')

## Section 5: Model Training

In [ ]:
# ---------------------------------------------------------------------------
# Loss function: CrossEntropyLoss with class weights
# The weight parameter upscales the loss contribution of minority class samples,
# mitigating the effect of class imbalance on gradient updates.
# ---------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimiser: Adam with L2 weight decay (equivalent to L2 regularisation)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=INIT_LR,
    weight_decay=1e-4   # L2 penalty to discourage excessively large weights
)

# Learning rate scheduler: reduce LR when validation loss plateaus.
# factor=0.5 halves the LR; patience=7 waits 7 epochs before reduction.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=7, min_lr=1e-7, verbose=True
)

# Early stopping monitor
early_stopper = EarlyStopping(patience=12, min_delta=1e-4)

# Path to save the best model checkpoint (lowest validation loss)
best_model_path = MODEL_DIR / 'cnn_best.pt'

# Training history — recorded per epoch for later visualisation
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------
print(f'{"="*70}')
print(f'  Training: {MODEL_NAME} (from scratch, no pretrained weights)')
print(f'  Device  : {DEVICE} | Max epochs: {MAX_EPOCHS} | Batch size: {BATCH_SIZE}')
print(f'{"="*70}')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>10} | {"Val Loss":>10} | {"Val Acc":>10} | {"LR":>10}')
print('-' * 70)

t_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    # --- Forward + backward pass over training set ---
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)

    # --- Evaluation on validation set (no gradient update) ---
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

    # --- Learning rate scheduler step ---
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    # --- Record epoch metrics ---
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # --- Check for improvement and save checkpoint ---
    improved = early_stopper(val_loss)
    marker   = ' <-- best' if improved else ''
    if improved:
        torch.save(model.state_dict(), best_model_path)

    # Log progress every epoch
    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc:>9.2%} | {val_loss:>10.4f} | {val_acc:>9.2%} | {current_lr:>10.2e}{marker}')

    # --- Early stopping check ---
    if early_stopper.should_stop:
        print(f'\nEarly stopping triggered at epoch {epoch} (patience={early_stopper.patience}).')
        break

elapsed = time.time() - t_start
epochs_trained = len(history['train_loss'])

print(f'{"="*70}')
print(f'Training complete.')
print(f'  Total epochs run : {epochs_trained}')
print(f'  Wall-clock time  : {elapsed/60:.1f} minutes')
print(f'  Best val loss    : {early_stopper.best_loss:.4f}')
print(f'  Best model saved : {best_model_path}')

# Persist training history to JSON for comparison in Notebook 07
with open(RES_DIR / f'{MODEL_NAME}_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'  History saved    : results/{MODEL_NAME}_history.json')

## Section 6: Learning Curves

The learning curves reveal training dynamics:
- A decreasing gap between training and validation curves indicates that regularisation is working
- A diverging gap (training continues to improve while validation plateaus) signals overfitting
- The vertical dashed line marks the epoch at which the best validation loss was recorded

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL_NAME} — Training History', fontsize=13, fontweight='bold')

epochs_ran  = range(1, epochs_trained + 1)
best_ep     = int(np.argmin(history['val_loss'])) + 1  # Epoch of lowest validation loss

# --- Panel A: Loss curves ---
ax = axes[0]
ax.plot(epochs_ran, history['train_loss'], 'b-', linewidth=1.5, label='Training Loss')
ax.plot(epochs_ran, history['val_loss'],   'r-', linewidth=1.5, label='Validation Loss', alpha=0.85)
ax.axvline(best_ep, color='green', linestyle='--', linewidth=1.5, alpha=0.8,
           label=f'Best val loss (epoch {best_ep})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Loss Curves')
ax.legend(fontsize=9)

# --- Panel B: Accuracy curves ---
ax = axes[1]
train_acc_pct = [v * 100 for v in history['train_acc']]
val_acc_pct   = [v * 100 for v in history['val_acc']]
ax.plot(epochs_ran, train_acc_pct, 'b-', linewidth=1.5, label='Training Accuracy')
ax.plot(epochs_ran, val_acc_pct,   'r-', linewidth=1.5, label='Validation Accuracy', alpha=0.85)
best_acc_ep = int(np.argmax(history['val_acc'])) + 1
ax.axvline(best_acc_ep, color='green', linestyle='--', linewidth=1.5, alpha=0.8,
           label=f'Best val acc (epoch {best_acc_ep})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy Curves')
ax.legend(fontsize=9)

# --- Panel C: Learning rate schedule ---
ax = axes[2]
ax.semilogy(epochs_ran, history['lr'], 'purple', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate (log scale)')
ax.set_title('Learning Rate Schedule')
ax.fill_between(epochs_ran, history['lr'], alpha=0.2, color='purple')

plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_learning_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Figure saved: results/{MODEL_NAME}_learning_curves.png')

## Section 7: Test Set Evaluation

The best checkpoint (lowest validation loss) is loaded before evaluation on the held-out test set.
The test set is used only once — here — to report unbiased final performance metrics.

In [ ]:
# Load the best weights identified during training
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

# Collect predictions, true labels, and softmax probabilities over the entire test set
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(DEVICE))
        # Softmax converts raw logits to class probabilities (required for ROC-AUC)
        probs  = torch.softmax(logits, dim=1)
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(y_batch.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

# Compute aggregate metrics
test_acc         = accuracy_score(all_labels, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
roc_auc          = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
total_params     = sum(p.numel() for p in model.parameters())

per_class_acc = [
    accuracy_score(all_labels[all_labels == i], all_preds[all_labels == i])
    for i in range(NUM_CLASSES)
]

print('=' * 52)
print(f'  {MODEL_NAME} — Final Test Set Results')
print('=' * 52)
print(f'  Accuracy  (weighted) : {test_acc * 100:.2f}%')
print(f'  Precision (weighted) : {prec * 100:.2f}%')
print(f'  Recall    (weighted) : {rec * 100:.2f}%')
print(f'  F1-Score  (weighted) : {f1 * 100:.2f}%')
print(f'  ROC-AUC   (OvR,wtd) : {roc_auc:.4f}')
print(f'  Total parameters     : {total_params:,}')
print(f'  Epochs trained       : {epochs_trained}')
print('=' * 52)
print('\nPer-Class Classification Report:')
print(classification_report(
    all_labels, all_preds,
    target_names=[c.replace('_', ' ').title() for c in CLASS_NAMES],
    digits=4
))

## Section 8: Confusion Matrix

In [ ]:
# Compute both the raw count matrix and the row-normalised (per-class recall) matrix
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
pretty  = [c.replace('_', ' ').title() for c in CLASS_NAMES]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'{MODEL_NAME} — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')

for ax, data, fmt, title, vmax in [
    (axes[0], cm,      'd',   'Absolute Counts',  None),
    (axes[1], cm_norm, '.2f', 'Normalised (Recall)', 1.0),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
        xticklabels=pretty, yticklabels=pretty,
        linewidths=0.5, linecolor='lightgray',
        vmin=0, vmax=vmax,
        annot_kws={'fontsize': 10, 'fontweight': 'bold'}
    )
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label',      fontsize=11)
    ax.set_title(title,              fontsize=11)
    ax.tick_params(axis='x', rotation=25)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

## Section 9: ROC Curves

In [ ]:
# Binarise labels for one-vs-rest ROC curve computation
y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(9, 7))
fig.suptitle(f'{MODEL_NAME} — ROC Curves (One-vs-Rest, Test Set)', fontsize=13, fontweight='bold')

for idx, (cls, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, idx], all_probs[:, idx])
    roc_auc_cls = auc(fpr, tpr)
    ax.plot(
        fpr, tpr,
        color=color, linewidth=2,
        label=f'{cls.replace("_", " ").title()} (AUC = {roc_auc_cls:.3f})'
    )

# Diagonal reference line: represents a random classifier (AUC = 0.5)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random classifier (AUC = 0.500)')
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
ax.set_ylabel('True Positive Rate (Sensitivity)',      fontsize=11)
ax.set_title(f'ROC Curves — Weighted AUC = {roc_auc:.4f}', fontsize=11)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])

plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()

## Section 10: Per-Class Accuracy

In [ ]:
per_class_acc_pct = [a * 100 for a in per_class_acc]
pretty = [c.replace('_', ' ').title() for c in CLASS_NAMES]

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle(f'{MODEL_NAME} — Per-Class Test Accuracy', fontsize=13, fontweight='bold')

bars = ax.bar(pretty, per_class_acc_pct, color=CLASS_COLORS, alpha=0.85, edgecolor='white')
ax.axhline(
    test_acc * 100, color='black', linestyle='--', linewidth=1.5,
    label=f'Overall accuracy ({test_acc * 100:.1f}%)'
)
# Annotate each bar with its value
for bar, val in zip(bars, per_class_acc_pct):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f'{val:.1f}%',
        ha='center', va='bottom',
        fontsize=10, fontweight='bold'
    )
ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(RES_DIR / f'{MODEL_NAME}_per_class_accuracy.png', bbox_inches='tight', dpi=150)
plt.show()

## Section 11: Save Metrics

In [ ]:
# Build and persist the full metrics dictionary.
# This file is consumed by 07_Model_Comparison.ipynb to build the final comparison table.
report_dict = classification_report(
    all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True
)

metrics = {
    'model_name':         MODEL_NAME,
    'architecture_type':  'Custom CNN (trained from scratch)',
    'test_accuracy':      float(test_acc),
    'weighted_precision': float(prec),
    'weighted_recall':    float(rec),
    'weighted_f1':        float(f1),
    'roc_auc_weighted':   float(roc_auc),
    'per_class_accuracy': {
        CLASS_NAMES[i]: float(per_class_acc[i]) for i in range(NUM_CLASSES)
    },
    'per_class_report':   report_dict,
    'total_params':       total_params,
    'epochs_trained':     epochs_trained,
    'fine_tuning':        False,
    'pretrained':         False,
    'hyperparameters': {
        'batch_size':    BATCH_SIZE,
        'init_lr':       INIT_LR,
        'optimizer':     'Adam',
        'weight_decay':  1e-4,
        'max_epochs':    MAX_EPOCHS,
        'scheduler':     'ReduceLROnPlateau(factor=0.5, patience=7)',
        'early_stopping_patience': 12,
        'gradient_clip_norm': 1.0,
    },
}

metrics_path = RES_DIR / f'{MODEL_NAME}_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Metrics saved to : {metrics_path}')
print(f'\nSummary: Accuracy={test_acc*100:.2f}% | F1={f1*100:.2f}% | '
      f'AUC={roc_auc:.4f} | Params={total_params:,}')